# Halide Perovskite — Bring-Up Path Validation

**Purpose:** validate device and PV connectivity before a campaign run.
This notebook is **strictly read-only** — it never moves a pump, never fires
the QEPro, and requires no queue server to be running.  Safe to run blind
at beamline setup time.

## Prerequisites

1. Python environment with `xpd-tools` installed:
   ```bash
   source ~/Documents/AutonomousExperiments/bnlvenv/bin/activate
   ```

2. EPICS channel-access reachable (you are on the NSLS2 network or VPN).
   PV prefixes are compiled into `build_halide_devices()` — no additional
   env vars are required for connectivity.

3. *(Optional)* A `.env` file in this directory with:
   ```
   QSERVER_HTTP_URI=https://xf28id2-xpd-qs1.nsls2.bnl.gov
   QSERVER_HTTP_SERVER_API_KEY=...
   ZMQ_CONSUMER_ADDR=ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock
   TILED_URI=https://tiled.nsls2.bnl.gov
   XPD_HALIDE_CONFIG=/path/to/configs/halide_beamline.json
   ```

## Run this notebook

```bash
cd ~/Documents/AutonomousExperiments/HalideExchange/uvvis
jupyter notebook validate_paths.ipynb
```

## After a successful run

- All devices connected  →  proceed to `halide_beamline_updated.ipynb`.
- Want to test actuation before the campaign:
  ```bash
  # Disconnect syringes from drive screws first!
  XPD_RUN_TRIAL=1 XPD_HALIDE_CONFIG=configs/halide_beamline.json \\
      pytest tests/optimization/test_campaign.py -k single_trial -v -s
  ```


---
## 1 — Environment

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Print env vars relevant to a campaign (informational only) ────────────
env_keys = [
    "QSERVER_HTTP_URI",
    "QSERVER_HTTP_SERVER_API_KEY",
    "ZMQ_CONSUMER_ADDR",
    "TILED_URI",
    "TILED_PROFILE",
    "XPD_HALIDE_CONFIG",
]

print("Environment variables:")
print("-" * 50)
for key in env_keys:
    val = os.environ.get(key, "<not set>")
    # Redact API keys — print first 6 chars only
    if "KEY" in key and val != "<not set>":
        val = val[:6] + "..." if len(val) > 6 else "***"
    print(f"  {key:<35} {val}")

config_path = os.environ.get("XPD_HALIDE_CONFIG", "configs/halide_beamline.json")
config_exists = Path(config_path).exists()
print()
print(f"  XPD_HALIDE_CONFIG exists on disk: {config_exists}")
if not config_exists:
    print("  (Config file missing — build one with halide_beamline_updated.ipynb)")

---
## 2 — Build device objects

`build_halide_devices()` instantiates all regular-ophyd devices with their
compiled-in PV prefixes.  It does **not** connect yet — connection happens
lazily when the next cell calls `check_halide_connections`.

In [ ]:
from xpd_tools.optimization.devices import (
    build_halide_devices,
    check_halide_connections,
    print_connection_report,
)

devices = build_halide_devices()    # mock=False → real EPICS PVs

print(f"Device keys ({len(devices)} total):")
for key, dev in sorted(devices.items()):
    print(f"  {key:<15}  {type(dev).__name__}")

---
## 3 — Connectivity probe

`check_halide_connections` calls `wait_for_connection` on every device and
does a `.get()` on the signals the acquisition plans depend on.  It is
**read-only**: no `set`, no `trigger`, no plan execution.

`print_connection_report` prints a coloured per-device summary and returns
`True` if everything connected, `False` otherwise.

In [ ]:
CONNECTION_TIMEOUT: float = 10.0   # seconds per device

reports = check_halide_connections(devices, timeout=CONNECTION_TIMEOUT)
all_ok: bool = print_connection_report(reports)

---
## 4 — Assert all paths are healthy

This cell raises if any device failed to connect so the notebook stops
with a clear error rather than silently continuing.

In [ ]:
failed = [
    key for key, report in reports.items()
    if not report.connected or report.failed_signals
]

if failed:
    print("\n❌  Failed devices:")
    for key in failed:
        r = reports[key]
        print(f"    {key}: connected={r.connected}, failed_signals={r.failed_signals}")
    raise RuntimeError(
        f"{len(failed)} device(s) did not connect: {failed}\n"
        "Check EPICS channel-access, PV prefixes, and IOC status."
    )

print("\n✓  All devices connected — paths look good.")
print("   Proceed to halide_beamline_updated.ipynb to start the campaign.")

---
## Next Steps

| Status | Next action |
|--------|-------------|
| ✓ All green | Open `halide_beamline_updated.ipynb` and start the campaign |
| ✗ Some failures | Check IOC status, EPICS network, and PV prefixes in `optimization/devices/__init__.py` |

### First actuation check (optional, before the campaign)

> ⚠️  **Disconnect DDS and Ultra syringes from their drive screws before running this.**

```bash
XPD_RUN_TRIAL=1 XPD_HALIDE_CONFIG=configs/halide_beamline.json \\
    pytest tests/optimization/test_campaign.py -k single_trial -v -s
```

This submits one point with small nonzero pump rates through the queue server
so you can visually confirm the drive screws are turning before committing
to a full campaign run.
